In [13]:
# 셀 1: 임포트 및 환경 설정
import os
import json
import torch
import multiprocessing as mp
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, default_data_collator, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset, Dataset
from datasets import logging as datasets_logging

datasets_logging.set_verbosity_warning()  # 경고 이상만 표시
mp.set_start_method("fork", force=True)
torch.set_num_threads(os.cpu_count())


In [14]:
# 셀 2: 설정 값 정의
MODEL_NAME = "/home/remote/Ai_Capstone_Project/SourceCode/polyglot-ko-3.8B"
DATA_FILE = "data/lora_data.jsonl"
OUTPUT_DIR = "/home/remote/Ai_Capstone_Project/AIbigdata_link/Fine-tuning-LoRA"
os.makedirs(OUTPUT_DIR, exist_ok=True)
MAX_LEN = 512


In [15]:
# 셀 3: JSONL 파일 로드 및 정보 확인 (디버깅용)
debug_ds = load_dataset(
    "json",
    data_files=DATA_FILE,
    split="train"
)
print(f"데이터셋 크기: {debug_ds.num_rows} rows")
print("컬럼명:", debug_ds.column_names)
print("피처 정보:", debug_ds.features)
print("첫 번째 예시:", debug_ds[0])

데이터셋 크기: 1705420 rows
컬럼명: ['instruction', 'response']
피처 정보: {'instruction': Value(dtype='string', id=None), 'response': {'text': Value(dtype='string', id=None), 'tags': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), 'intensity': Value(dtype='int64', id=None)}}
첫 번째 예시: {'instruction': '캐릭터 설명: 이 캐릭터는 고대 시대의 중년 귀족 출신 학자입니다. 침착함 성격을 지녔습니다.\n상황: 플레이어에게 새로운 임무나 퀘스트를 제안하는 상황입니다.', 'response': {'text': '반드시 해낼 수 있을 거야!', 'tags': ['희망', '확언'], 'intensity': 5}}


In [16]:
# 셀 4: 양자화 설정 및 토크나이저, 모델 로드
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)
model.gradient_checkpointing_enable()
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

In [17]:
# 셀 5: LoRA 어댑터 설정 및 파라미터 수 확인
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["attention.dense"],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)
model = get_peft_model(model, lora_config)
# LoRA 파라미터 수
total_params = sum(p.numel() for p in model.parameters())
lora_params = sum(p.numel() for n,p in model.named_parameters() if p.requires_grad and 'lora_' in n)
print(f"모델 전체 파라미터 수: {total_params}")
print(f"LoRA 어댑터 파라미터 수: {lora_params} ({lora_params/total_params*100:.2f}% 비율)")

모델 전체 파라미터 수: 3811547136
LoRA 어댑터 파라미터 수: 1572864 (0.04% 비율)


In [18]:
# 셀 6: 패딩 토큰 설정
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

In [19]:
# 셀 7: JSONL 스트리밍 + map 기반 전처리
def process_sample(sample):
    if 'instruction' in sample and 'output' in sample:
        prompt = f"### 질문: {sample['instruction']}\n\n### 답변: "
        answer = sample['output'] + tokenizer.eos_token
        enc = tokenizer(
            prompt + answer,
            truncation=True,
            max_length=MAX_LEN,
            padding='max_length'
        )
        p_ids = tokenizer(prompt, add_special_tokens=False).input_ids
        a_ids = tokenizer(answer, add_special_tokens=False).input_ids
        labels = [-100] * len(p_ids) + a_ids
        labels = labels[:MAX_LEN] + [-100] * (MAX_LEN - len(labels))
        enc['labels'] = labels
    elif 'text' in sample:
        enc = tokenizer(sample['text'], truncation=True, max_length=MAX_LEN, padding='max_length')
        enc['labels'] = enc['input_ids'].copy()
    else:
        merged = ' '.join(map(str, sample.values()))
        enc = tokenizer(merged, truncation=True, max_length=MAX_LEN, padding='max_length')
        enc['labels'] = enc['input_ids'].copy()
    return enc

with open(DATA_FILE, 'r', encoding='utf-8') as f:
    ds_len = sum(1 for _ in f)
raw_ds = load_dataset('json', data_files=DATA_FILE, streaming=True)['train']
train_dataset = raw_ds.map(
    process_sample,
    batched=False,
    remove_columns=raw_ds.column_names
).with_format('torch')

In [20]:
# 셀 8: 디버깅용 샘플 확인 (스트리밍 대응)
train_iter = iter(train_dataset)
for idx in range(3):
    sample = next(train_iter)
    input_ids = sample['input_ids']
    labels = sample['labels']
    text = tokenizer.decode(input_ids, skip_special_tokens=True)
    decoded = [tokenizer.decode([lid], skip_special_tokens=True) for lid in labels if lid != -100]
    print(f"=== 샘플 {idx} ===\n{text}\n레이블토큰:{decoded}\n")

=== 샘플 0 ===
캐릭터 설명: 이 캐릭터는 고대 시대의 중년 귀족 출신 학자입니다. 침착함 성격을 지녔습니다.
상황: 플레이어에게 새로운 임무나 퀘스트를 제안하는 상황입니다. {'text': '반드시 해낼 수 있을 거야!', 'tags': ['희망', '확언'], 'intensity': 5}
레이블토큰:['캐릭터', ' 설명', ':', ' 이', ' 캐릭터', '는', ' 고대', ' 시대', '의', ' 중년', ' 귀족', ' 출신', ' 학자', '입니다', '.', ' 침착', '함', ' 성격', '을', ' 지녔', '습니다', '.', '\n', '상황', ':', ' 플레이어', '에게', ' 새로운', ' 임무', '나', ' 퀘', '스트', '를', ' 제안', '하', '는', ' 상황', '입니다', '.', ' ', '{', "'", 't', 'ex', 't', "'", ':', " '", '반드시', ' 해낼', ' 수', ' 있', '을', ' 거', '야', '!', "',", " '", 't', 'ag', 's', "'", ':', ' [', "'", '희망', "',", " '", '확', '언', "'", ']', ',', " '", 'int', 'ens', 'ity', "'", ':', ' 5', '}', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', ''

In [21]:
# 셀 9: TrainingArguments 설정 (에폭당 2000 스텝 제한)
batch_size = 4
accum_steps = 4
#epochs = 2
epochs = 5
#steps_per_epoch = 150  # 에폭당 업데이트 스텝 수 제한
steps_per_epoch = 2000  # 에폭당 업데이트 스텝 수 제한
max_steps = steps_per_epoch * epochs

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=accum_steps,
    learning_rate=2e-4,
    fp16=True,
    optim='adamw_torch',
    dataloader_num_workers=2,
    logging_steps=100,                # 100 스텝마다 로깅
    save_steps=steps_per_epoch // 2,   # 에폭당 2회 체크포인트 (2000/2)
    save_strategy='steps',
    save_total_limit=3,               # 최근 3개만 보관
    logging_dir=f"{OUTPUT_DIR}/logs",
    report_to='tensorboard',
    num_train_epochs=epochs,
    max_steps=max_steps
)

In [ ]:
# 셀 10: Trainer 초기화 및 학습 시작 및 학습 시작
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=default_data_collator,
    tokenizer=tokenizer
)
trainer.train()


/tmp/ipykernel_578774/847913730.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Too many dataloader workers: 2 (max is dataset.n_shards=1). Stopping 1 dataloader workers.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid d

Step,Training Loss


In [ ]:
# 셀 11: 학습 완료 후 모델·토크나이저 저장
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("✅ 모델과 토크나이저가 저장되었습니다:", OUTPUT_DIR)


✅ 모델과 토크나이저가 저장되었습니다: /home/remote/Ai_Capstone_Project/AIbigdata_link/Fine-tuning-LoRA


In [ ]:
# 셀 12: 메모리 정리 및 캐시 해제
import gc

# 주요 객체 삭제
del trainer
del model
# 추가로 필요 없어진 변수 삭제
try:
    del train_dataset
    del raw_ds
    del debug_ds
except NameError:
    pass

# 가비지 컬렉션 및 CUDA 캐시 비우기
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✅ CUDA 캐시를 비웠습니다")
print("✅ 메모리 정리 완료")


✅ CUDA 캐시를 비웠습니다
✅ 메모리 정리 완료
